# PyTorch CNN Common Syntax

## 1. Basic Imports

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
```

Commonly used PyTorch modules:

```python
nn.Conv2d
nn.BatchNorm2d
nn.ReLU
nn.MaxPool2d
nn.AvgPool2d
nn.AdaptiveAvgPool2d
nn.Flatten
nn.Linear
nn.Dropout
```

## 2. PyTorch Image Shape Convention

PyTorch usually uses the **channels-first** image format:

```text
(batch, channels, height, width)
```

For a batch of RGB images with size `64 x 64`:

```python
x.shape = (batch_size, 3, 64, 64)
```

Comparison with TensorFlow/Keras:

```text
TensorFlow / Keras: (batch, height, width, channels)
PyTorch:            (batch, channels, height, width)
```

So the same RGB image has:

```text
TensorFlow input shape: (64, 64, 3)
PyTorch input shape:    (3, 64, 64)
```

## 3. `nn.Sequential`

Use `nn.Sequential` when the model is a simple straight chain.

```python
model = nn.Sequential(
    nn.Conv2d(
        in_channels=3,
        out_channels=32,
        kernel_size=3,
        stride=1,
        padding=1
    ),
    nn.BatchNorm2d(32),
    nn.ReLU(),

    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Flatten(),
    nn.Linear(32 * 32 * 32, 1),
    nn.Sigmoid()
)
```

Input shape:

```text
(batch, 3, 64, 64)
```

Shape flow:

```text
(batch, 3, 64, 64)
-> Conv2d
(batch, 32, 64, 64)
-> MaxPool2d
(batch, 32, 32, 32)
-> Flatten
(batch, 32768)
-> Linear
(batch, 1)
-> Sigmoid
(batch, 1)
```

## 4. `nn.Module` Style

The most common way to build models in PyTorch is to subclass `nn.Module`.

```python
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=32,
            kernel_size=3,
            stride=1,
            padding=1
        )

        self.bn1 = nn.BatchNorm2d(32)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten()
        self.fc = nn.Linear(32 * 32 * 32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.flatten(x)
        x = self.fc(x)
        x = self.sigmoid(x)

        return x
```

Create the model:

```python
model = SimpleCNN()
```

In PyTorch:

```text
__init__ defines the layers.
forward defines how data flows through the layers.
```

## 5. Conv2d

Basic syntax:

```python
nn.Conv2d(
    in_channels=3,
    out_channels=32,
    kernel_size=3,
    stride=1,
    padding=1
)
```

Shorter version:

```python
nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
```

Parameter meanings:

| Parameter | Meaning |
|---|---|
| `in_channels` | Number of input channels |
| `out_channels` | Number of filters, also the number of output channels |
| `kernel_size` | Filter size |
| `stride` | Step size of the filter |
| `padding` | Padding size |
| `bias` | Whether to use bias |

Example:

```python
nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
```

Input:

```text
(batch, 3, 64, 64)
```

Output:

```text
(batch, 32, 64, 64)
```

Because:

```text
kernel_size = 3
padding = 1
stride = 1
```

keeps the spatial size unchanged.

## 6. Padding in PyTorch

In PyTorch, padding is usually written directly inside `nn.Conv2d`.

For a `3 x 3` kernel with stride `1`:

```python
nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
```

For a `5 x 5` kernel with stride `1`:

```python
nn.Conv2d(3, 32, kernel_size=5, stride=1, padding=2)
```

For a `7 x 7` kernel with stride `1`:

```python
nn.Conv2d(3, 32, kernel_size=7, stride=1, padding=3)
```

Rule of thumb for odd kernel sizes and stride `1`:

```text
padding = (kernel_size - 1) / 2
```

PyTorch also supports:

```python
nn.Conv2d(3, 32, kernel_size=3, stride=1, padding="same")
```

However, when learning CNN fundamentals, writing explicit integer padding is often clearer:

```python
padding=1
```

## 7. ZeroPad2d

PyTorch equivalent of TensorFlow `ZeroPadding2D`:

```python
nn.ZeroPad2d(3)
```

Example:

```python
model = nn.Sequential(
    nn.ZeroPad2d(3),
    nn.Conv2d(
        in_channels=3,
        out_channels=32,
        kernel_size=7,
        stride=1,
        padding=0
    )
)
```

Shape flow:

```text
(batch, 3, 64, 64)
-> ZeroPad2d(3)
(batch, 3, 70, 70)
-> Conv2d 7x7 valid
(batch, 32, 64, 64)
```

But in practice, this is usually written more simply as:

```python
nn.Conv2d(
    in_channels=3,
    out_channels=32,
    kernel_size=7,
    stride=1,
    padding=3
)
```

So the common style is:

```text
Use padding inside Conv2d unless you need manual or asymmetric padding.
```

## 8. Batch Normalization

In TensorFlow/Keras:

```python
layers.BatchNormalization(axis=-1)
```

In PyTorch:

```python
nn.BatchNorm2d(num_features=32)
```

or shorter:

```python
nn.BatchNorm2d(32)
```

For CNNs, PyTorch tensors have shape:

```text
(batch, channels, height, width)
```

So:

```python
nn.BatchNorm2d(32)
```

means:

```text
Normalize 32 channels.
```

Common pattern:

```python
nn.Conv2d(3, 32, kernel_size=3, padding=1),
nn.BatchNorm2d(32),
nn.ReLU()
```

Key comparison:

```text
TensorFlow BatchNorm: BatchNormalization(axis=-1)
PyTorch BatchNorm:    BatchNorm2d(num_channels)
```

## 9. ReLU / Activation

Layer style:

```python
nn.ReLU()
```

Functional style:

```python
F.relu(x)
```

Example inside `nn.Module`:

```python
x = self.relu(x)
```

or:

```python
x = F.relu(x)
```

If you define ReLU as a layer:

```python
self.relu = nn.ReLU()
```

then use it in `forward`:

```python
x = self.relu(x)
```

## 10. Pooling

### Max Pooling

```python
nn.MaxPool2d(kernel_size=2, stride=2)
```

Shorter version:

```python
nn.MaxPool2d(2)
```

Shape example:

```text
(batch, 32, 64, 64)
-> MaxPool2d(2)
(batch, 32, 32, 32)
```

Pooling reduces height and width, but keeps channels unchanged.

### Average Pooling

```python
nn.AvgPool2d(kernel_size=2, stride=2)
```

### Global Average Pooling

In PyTorch, global average pooling is usually written as:

```python
nn.AdaptiveAvgPool2d((1, 1))
```

Example:

```text
(batch, 128, 8, 8)
-> AdaptiveAvgPool2d((1, 1))
(batch, 128, 1, 1)
-> Flatten
(batch, 128)
```

## 11. Flatten

Layer style:

```python
nn.Flatten()
```

Tensor operation style:

```python
x = torch.flatten(x, start_dim=1)
```

Why `start_dim=1`?

```text
dim=0 is the batch dimension.
We usually keep the batch dimension unchanged.
```

Example:

```text
(batch, 32, 32, 32)
-> torch.flatten(x, start_dim=1)
(batch, 32768)
```

## 12. Linear Layer

TensorFlow/Keras:

```python
layers.Dense(128)
```

PyTorch:

```python
nn.Linear(in_features, out_features)
```

Hidden layer:

```python
nn.Linear(32768, 128)
```

Binary classification output:

```python
nn.Linear(128, 1)
```

Multi-class classification output:

```python
nn.Linear(128, num_classes)
```

Conceptually:

```text
Linear layer = affine transformation
```

```text
output = input @ weight.T + bias
```

## 13. Dropout

```python
nn.Dropout(p=0.5)
```

Example:

```python
self.dropout = nn.Dropout(0.5)
```

Use in `forward`:

```python
x = self.dropout(x)
```

Dropout behaves differently in training and evaluation:

```text
model.train() enables dropout.
model.eval() disables dropout.
```

## 14. Binary Classification

There are two common approaches.

### Approach 1: Model Outputs Sigmoid Probability

```python
class BinaryCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
```

Loss:

```python
criterion = nn.BCELoss()
```

Prediction:

```python
probs = model(x)
preds = (probs > 0.5).long()
```

### Approach 2: Model Outputs Raw Logits

This approach is usually preferred because it is more numerically stable.

```python
class BinaryCNNLogits(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
```

Loss:

```python
criterion = nn.BCEWithLogitsLoss()
```

Prediction:

```python
logits = model(x)
probs = torch.sigmoid(logits)
preds = (probs > 0.5).long()
```

Important rule:

```text
If you use BCEWithLogitsLoss, do not put Sigmoid inside the model.
```

## 15. Multi-Class Classification

For multi-class classification, the model usually outputs raw logits.

```python
class MultiClassCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
```

Loss:

```python
criterion = nn.CrossEntropyLoss()
```

Prediction:

```python
logits = model(x)
preds = torch.argmax(logits, dim=1)
```

Important rule:

```text
If you use CrossEntropyLoss, do not apply Softmax inside the model.
```

## 16. CNN with Global Average Pooling

Modern CNNs often use global average pooling to reduce the number of parameters in the classifier head.

```python
class GAPCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.classifier(x)
        return x
```

Shape flow:

```text
(batch, 3, 64, 64)
-> features
(batch, 128, 16, 16)
-> AdaptiveAvgPool2d((1, 1))
(batch, 128, 1, 1)
-> flatten
(batch, 128)
-> Linear
(batch, num_classes)
```

## 17. Device Syntax: CPU / GPU

Create device:

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

Move model to device:

```python
model = model.to(device)
```

Move data to device:

```python
images = images.to(device)
labels = labels.to(device)
```

Common pattern:

```python
for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)
```

Important rule:

```text
The model and the data must be on the same device.
```

## 18. DataLoader

PyTorch usually uses `Dataset` and `DataLoader` for batching data.

```python
from torch.utils.data import DataLoader
```

Training loader:

```python
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)
```

Validation loader:

```python
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)
```

Conceptually:

```text
Dataset gives individual samples.
DataLoader groups samples into batches.
```

## 19. Training Loop

Unlike Keras, PyTorch usually uses an explicit training loop.

```python
model.train()

 for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)

    logits = model(images)
    loss = criterion(logits, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

Correct version without indentation mistake:

```python
model.train()

for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)

    logits = model(images)
    loss = criterion(logits, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

Meaning:

```text
model.train()          enables training mode
logits = model(images) forward pass
loss = criterion(...)  computes loss
optimizer.zero_grad()  clears old gradients
loss.backward()        backpropagation
optimizer.step()       updates weights
```

## 20. Evaluation Loop

```python
model.eval()

total_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

avg_loss = total_loss / total
accuracy = correct / total
```

Important:

```python
model.eval()
```

turns off training behavior for layers such as Dropout and changes BatchNorm behavior.

```python
with torch.no_grad():
```

disables gradient tracking and saves memory during validation or testing.

## 21. Optimizer Syntax

### Adam

```python
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
```

### SGD

```python
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9
)
```

### AdamW

```python
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)
```

## 22. Save and Load Model

Save model weights:

```python
torch.save(model.state_dict(), "model.pth")
```

Load model weights:

```python
model = MultiClassCNN(num_classes=10)
model.load_state_dict(torch.load("model.pth", map_location=device))
model = model.to(device)
model.eval()
```

Common file extensions:

```text
.pt
.pth
```

## 23. PyTorch Version of HappyModel

Original architecture:

```text
ZEROPAD2D -> CONV2D -> BATCHNORM -> RELU -> MAXPOOL -> FLATTEN -> DENSE
```

PyTorch version with explicit `ZeroPad2d`:

```python
class HappyModelTorch(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.ZeroPad2d(3),
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=7,
                stride=1,
                padding=0
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)
```

Input shape:

```text
(batch, 3, 64, 64)
```

Shape flow:

```text
(batch, 3, 64, 64)
-> ZeroPad2d(3)
(batch, 3, 70, 70)
-> Conv2d 7x7
(batch, 32, 64, 64)
-> BatchNorm2d
(batch, 32, 64, 64)
-> ReLU
(batch, 32, 64, 64)
-> MaxPool2d
(batch, 32, 32, 32)
-> Flatten
(batch, 32768)
-> Linear
(batch, 1)
-> Sigmoid
(batch, 1)
```

More common version without explicit `ZeroPad2d`:

```python
class HappyModelTorchCommon(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=7,
                stride=1,
                padding=3
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)
```

## 24. PyTorch vs TensorFlow Syntax Mapping

| Concept | TensorFlow / Keras | PyTorch |
|---|---|---|
| Input shape | `(H, W, C)` | `(C, H, W)` |
| Batch shape | `(m, H, W, C)` | `(m, C, H, W)` |
| Conv | `layers.Conv2D(filters, kernel_size)` | `nn.Conv2d(in_channels, out_channels, kernel_size)` |
| BatchNorm CNN | `BatchNormalization(axis=-1)` | `nn.BatchNorm2d(num_channels)` |
| ReLU | `layers.ReLU()` | `nn.ReLU()` |
| MaxPool | `layers.MaxPool2D(2)` | `nn.MaxPool2d(2)` |
| Flatten | `layers.Flatten()` | `nn.Flatten()` or `torch.flatten(x, 1)` |
| Dense | `layers.Dense(units)` | `nn.Linear(in_features, out_features)` |
| Sequential | `keras.Sequential([...])` | `nn.Sequential(...)` |
| Model class | `keras.Model` | `nn.Module` |
| Training | `model.fit(...)` | Manual training loop |
| Prediction | `model.predict(...)` | `model(x)` |
| Device | Often automatic | Explicit `.to(device)` |

## 25. Common Mistakes

| Mistake | Correct |
|---|---|
| Using input shape `(batch, H, W, C)` in PyTorch | Use `(batch, C, H, W)` |
| Writing `Conv2d(32, 3)` like Keras | Use `Conv2d(in_channels, out_channels, kernel_size)` |
| Forgetting `forward()` | Define `forward(self, x)` |
| Applying Softmax before `CrossEntropyLoss` | Output logits directly |
| Applying Sigmoid before `BCEWithLogitsLoss` | Output logits directly |
| Forgetting `optimizer.zero_grad()` | Call it before `loss.backward()` |
| Forgetting `model.train()` / `model.eval()` | Set the correct mode |
| Forgetting `.to(device)` | Move model and data to the same device |
| Wrong `Linear` input size | Check tensor shape before `nn.Linear` |
| Forgetting `torch.no_grad()` in evaluation | Use it during validation or testing |

## 26. Short Cheat Sheet

```python
import torch
import torch.nn as nn
```

```python
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.classifier(x)
        return x
```

Training pattern:

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()

for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)

    logits = model(images)
    loss = criterion(logits, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

## 27. Final Memory

```text
TensorFlow / Keras hides the training loop with model.fit().
PyTorch usually makes the training loop explicit.

TensorFlow image shape: (batch, height, width, channels)
PyTorch image shape:    (batch, channels, height, width)

Keras Conv2D: filters first.
PyTorch Conv2d: in_channels, out_channels, kernel_size.

Keras BatchNorm: BatchNormalization(axis=-1).
PyTorch BatchNorm2d: BatchNorm2d(num_channels).
```
